<div align="center">

### RR Skillverse — Free Learning Handbook
**by Raushan Ranjan**

*A personal educational reference for structured learning and hands-on practice. Shared for learning purposes only — not a commercial product or paid service.*

</div>

---

# Module 10 — MLOps & On-Premises Deployment

**AI & Machine Learning: Advanced Engineering with Cybersecurity — 5-Day Program**

*Module 10 of 12 · 3 hours · Continues the RR Finance system built in Modules 1–9*

## Recap — what RR Finance already has

Nine modules have built real, tested models: a baseline classifier, an ANN, a CNN, a language model, RAG and agents, explainability tooling, federated and privacy-preserving training, multimodal pipelines, and graph-based fraud detection. **Every one of them, so far, has only ever run inside a Jupyter notebook.** None of them has ever answered a real HTTP request, run inside a container, or been deployed anywhere a real applicant's request could actually reach it.

Module 10 closes that gap. On-premises specifically, not cloud -- because RR Finance's own data (applicant PII, financial records) is exactly the kind of data Module 7's privacy work and typical financial-sector regulation both push toward keeping inside an institution's own infrastructure, not a third-party cloud. This module wraps Module 1's real saved pipeline in a real, running API, containerizes it, writes real Kubernetes manifests for it, scans it for real misconfigurations with Trivy, and secures it with real X.509 mutual TLS -- the same "attempt the real tool, disclose honestly when this sandbox can't reach something, build a real working alternative" discipline this course has followed since Module 8.

In [ ]:
from pathlib import Path
import json

Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)
Path("deployment").mkdir(exist_ok=True)

print("Project folders ready: data/, artifacts/, deployment/")

In [ ]:
%pip install -q fastapi uvicorn pydantic hvac cryptography joblib pandas scikit-learn requests
print("Setup complete -- if you saw 'Requirement already satisfied' lines above, that is expected and fine.")
print("\nThis module ALSO uses real external tools this notebook cannot pip-install:")
print("  - Trivy (security scanner): downloaded directly from its GitHub release in the setup cell below")
print("  - OpenSSL (TLS certificates): a standard system tool, already present on virtually every Linux machine")

---
## Lesson 1 — Recap: loading what Module 1 actually built

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Confirm RR Finance's real trained model is available before wrapping it in a deployable service. |
| **2. Why does it matter in finance?** | This module deploys the EXACT model Modules 1 and 6 validated and audited -- not a placeholder, and not a retrained stand-in. |
| **3. Why this technique?** | Load Module 1's real saved pipeline directly, exactly as every module since Module 2 has. |
| **4. What do the parameters mean?** | N/A -- this is a load step. |
| **5. What is happening mathematically?** | N/A. |
| **6. What happens if we change it?** | If Module 1's artifact is missing, this module cannot proceed -- deployment has nothing to deploy without it.

In [ ]:
import joblib

BASELINE_PATH = Path("artifacts/baseline_logreg_pipeline.joblib")
MODULE1_METRICS_PATH = Path("artifacts/module1_metrics.json")

if not BASELINE_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {BASELINE_PATH.resolve()}.\n"
        "Run Module 1's notebook first -- this module deploys its REAL saved pipeline, not a fresh one."
    )

pipeline = joblib.load(BASELINE_PATH)
print("Loaded Module 1's ACTUAL saved pipeline:", BASELINE_PATH.resolve())

if MODULE1_METRICS_PATH.exists():
    with open(MODULE1_METRICS_PATH) as f:
        module1_metrics = json.load(f)
    print(f"Test AUC: {module1_metrics['baseline_logreg_test_auc']:.4f} on {module1_metrics['dataset_rows']} applicants")
    print("\nThis is the model Module 10 deploys -- everything below wraps THIS exact object, unchanged.")

---
## Lesson 2 — Serving the Model with FastAPI

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | A trained model sitting in a notebook cannot answer a real HTTP request from a loan-origination system. It needs to run as a service. |
| **2. Why does it matter in finance?** | This is the literal boundary between "a data science experiment" and "a system a real application depends on" -- every module's model has been on the experiment side of that line until now. |
| **3. Why this technique?** | **FastAPI**, paired with **Pydantic** for request validation, is a real, widely-used Python framework for exactly this: type-checked, auto-documented, fast HTTP APIs -- run here by **uvicorn**, a real ASGI server. |
| **4. What do the parameters mean?** | Each `LoanApplication` field's `Field(..., ge=..., le=...)` constraints are REAL validation rules -- e.g. `credit_score` must be between 300 and 850, matching the real range Module 1's data used -- enforced automatically before the request ever reaches the model. |
| **5. What is happening mathematically?** | No new modelling -- this lesson wraps Module 1's already-fitted pipeline unchanged; the only new code is the HTTP interface around it. |
| **6. What happens if we change it?** | This lesson runs a REAL server and sends it REAL HTTP requests from within this notebook -- including a deliberately invalid one, to show the validation layer actually rejecting bad input before it ever reaches the model.

In [ ]:
api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import pandas as pd

app = FastAPI(title="RR Finance Loan Risk API", version="1.0")
pipeline = joblib.load("baseline_logreg_pipeline.joblib")

FEATURES = [
    "annual_income", "monthly_debt", "loan_amount", "loan_term_months",
    "credit_score", "employment_years", "account_age_months",
    "num_previous_loans", "previous_defaults", "debt_to_income", "loan_to_income",
]

class LoanApplication(BaseModel):
    annual_income: float = Field(..., gt=0)
    monthly_debt: float = Field(..., ge=0)
    loan_amount: float = Field(..., gt=0)
    loan_term_months: int = Field(..., gt=0)
    credit_score: int = Field(..., ge=300, le=850)
    employment_years: float = Field(..., ge=0)
    account_age_months: float = Field(..., ge=0)
    num_previous_loans: int = Field(..., ge=0)
    previous_defaults: int = Field(..., ge=0)
    debt_to_income: float = Field(..., ge=0)
    loan_to_income: float = Field(..., ge=0)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(application: LoanApplication):
    row = pd.DataFrame([application.model_dump()])[FEATURES]
    prob = pipeline.predict_proba(row)[0, 1]
    decision = "REJECT" if prob >= 0.5 else "APPROVE"
    return {"default_probability": round(float(prob), 4), "decision": decision}
'''

Path("deployment/app.py").write_text(api_code)
import shutil
shutil.copy(BASELINE_PATH, "deployment/baseline_logreg_pipeline.joblib")
print("Wrote deployment/app.py and copied the real pipeline artifact next to it.")
print(api_code)

In [ ]:
import subprocess, time, requests, os, signal

# Start the REAL FastAPI server as a subprocess, exactly as it would run in production.
server_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="deployment", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(3)  # give uvicorn a moment to bind the port

health = requests.get("http://127.0.0.1:8000/health")
print("GET /health ->", health.status_code, health.json())

In [ ]:
# A REAL prediction request against the REAL running server
valid_application = {
    "annual_income": 45000, "monthly_debt": 1800, "loan_amount": 20000, "loan_term_months": 36,
    "credit_score": 610, "employment_years": 2, "account_age_months": 18, "num_previous_loans": 2,
    "previous_defaults": 1, "debt_to_income": 0.48, "loan_to_income": 0.44,
}
response = requests.post("http://127.0.0.1:8000/predict", json=valid_application)
print("POST /predict (valid application) ->", response.status_code)
print(response.json())

# A deliberately INVALID request -- credit_score out of the real 300-850 range
invalid_application = dict(valid_application, credit_score=9999)
bad_response = requests.post("http://127.0.0.1:8000/predict", json=invalid_application)
print("\nPOST /predict (invalid credit_score=9999) ->", bad_response.status_code)
print(bad_response.json())

server_proc.terminate()
server_proc.wait()
print("\nServer stopped.")

**Reading this honestly:** this is a real server, started as a real subprocess, answering real HTTP requests -- not a mocked demonstration. The valid request returns the exact same prediction Module 1's pipeline would give in a notebook; the invalid request is rejected with a `422` status and a precise error message identifying exactly which field failed and why, BEFORE it ever reaches the model. That validation boundary is not optional in production -- a model has no way to sanity-check a credit score of 9999 on its own.

---
## Lesson 3 — Containerizing the Service with Docker

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Lesson 2's server depends on the exact Python version, packages, and file layout of THIS machine. A container packages all of that into one portable, reproducible unit. |
| **2. Why does it matter in finance?** | On-premises deployment means RR Finance's own infrastructure team runs this service, on their own servers -- a container is what makes "works on my machine" into "works on their machine" too. |
| **3. Why this technique?** | **Docker** is the standard containerization tool; a `Dockerfile` is a real, versioned build recipe -- not a deployment script that only works once. |
| **4. What do the parameters mean?** | `FROM python:3.11-slim` pins the exact base image; `COPY` and `RUN pip install` build the image in reproducible layers; `EXPOSE 8000` documents the port the container listens on. |
| **5. What is happening mathematically?** | N/A -- this is systems engineering, not a modelling technique. |
| **6. What happens if we change it?** | This notebook cannot run an actual Docker daemon (no privileged container-in-container access in this sandbox) -- so this lesson writes a REAL, valid Dockerfile, and Lesson 4 scans it with a REAL security scanner, rather than pretending to run `docker build` when it can't.

In [ ]:
dockerfile_content = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY baseline_logreg_pipeline.joblib .

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=3s CMD python3 -c "import requests; requests.get('http://localhost:8000/health').raise_for_status()"

RUN useradd --create-home --shell /bin/bash apiuser
USER apiuser

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

requirements_content = "fastapi\nuvicorn\npydantic\njoblib\npandas\nscikit-learn\n"

Path("deployment/Dockerfile").write_text(dockerfile_content)
Path("deployment/requirements.txt").write_text(requirements_content)
print("Wrote deployment/Dockerfile and deployment/requirements.txt\n")
print(dockerfile_content)

**Reading this honestly:** notice this Dockerfile already includes a non-root `USER` and a `HEALTHCHECK` -- Lesson 4 shows exactly why those two lines matter, by scanning an EARLIER, less careful version of this same Dockerfile and finding real, specific problems.

---
## Lesson 4 — DevSecOps Scanning with Trivy

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | A Dockerfile that LOOKS reasonable can still contain real, well-known security misconfigurations -- running as root, missing a health check -- that a human reviewer might miss but an automated scanner won't. |
| **2. Why does it matter in finance?** | This is the automated gate a real CI/CD pipeline would run before ANY container reaches RR Finance's production infrastructure -- catching exactly the kind of mistake that turns into a real incident. |
| **3. Why this technique?** | **Trivy**, downloaded here directly from its real GitHub release, is a widely-used open-source security scanner covering vulnerabilities, misconfigurations, and secrets across containers, filesystems, and infrastructure-as-code. |
| **4. What do the parameters mean?** | `trivy config <dir>` scans Dockerfiles, Kubernetes manifests, and similar config files for misconfigurations, using Trivy's built-in policy checks. |
| **5. What is happening mathematically?** | N/A -- these are rule-based policy checks (similar in spirit to a linter), not a learned model. |
| **6. What happens if we change it?** | This sandbox cannot reach `mirror.gcr.io`, the host Trivy normally pulls its latest policy bundle from -- Trivy handles this gracefully by falling back to its EMBEDDED checks, which is exactly what happens below, disclosed honestly rather than hidden.

In [ ]:
import subprocess

trivy_path = Path("trivy")
if not trivy_path.exists():
    print("Downloading the real Trivy binary from its GitHub release...")
    subprocess.run(
        ["curl", "-sL",
         "https://github.com/aquasecurity/trivy/releases/download/v0.74.0/trivy_0.74.0_Linux-64bit.tar.gz",
         "-o", "trivy.tar.gz"], check=True,
    )
    subprocess.run(["tar", "xzf", "trivy.tar.gz"], check=True)

version_check = subprocess.run(["./trivy", "--version"], capture_output=True, text=True)
print(version_check.stdout)

In [ ]:
# A NAIVE, less careful Dockerfile -- the kind a first draft might actually look like,
# BEFORE the fixes already baked into Lesson 3's real deployment/Dockerfile.
naive_dockerfile = '''FROM python:3.11
WORKDIR /app
COPY . /app
RUN pip install -r requirements.txt
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

Path("deployment_naive").mkdir(exist_ok=True)
Path("deployment_naive/Dockerfile").write_text(naive_dockerfile)

print("=== Scanning the NAIVE Dockerfile (before Lesson 3's fixes) ===\n")
result = subprocess.run(["./trivy", "config", "deployment_naive"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
print("=== Scanning Lesson 3's REAL Dockerfile (deployment/Dockerfile, already fixed) ===\n")
result_fixed = subprocess.run(["./trivy", "config", "deployment"], capture_output=True, text=True)
print(result_fixed.stdout)

**Reading this honestly:** the naive Dockerfile fails Trivy's real, built-in checks -- typically flagging the missing non-root `USER` (a real, common HIGH-severity finding: running a container as root means a container-escape vulnerability hands an attacker root on the host too) and the missing `HEALTHCHECK`. Lesson 3's actual deployment Dockerfile, which already included both fixes, passes clean. This is precisely the before/after discipline this course has used since Module 1's poisoning defence: a real problem, found by a real tool, fixed by a real, specific change -- not a demonstration staged to look clean from the start.

---
## Lesson 5 — Kubernetes Manifests

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | One running container is a single point of failure. RR Finance needs the service to survive a crashed pod, scale under load, and be reachable through a stable internal address. |
| **2. Why does it matter in finance?** | A loan-decision API going down because one process crashed is a real availability incident -- Kubernetes' whole purpose is making that NOT happen. |
| **3. Why this technique?** | **Kubernetes** is the standard container-orchestration platform; a `Deployment` manages replica pods and restarts failed ones, a `Service` gives them one stable internal DNS name. |
| **4. What do the parameters mean?** | `replicas: 3` runs three copies for redundancy; `livenessProbe`/`readinessProbe` point at the SAME `/health` endpoint Lesson 2 already built and tested; `resources.limits` caps how much CPU/memory one pod can consume. |
| **5. What is happening mathematically?** | N/A -- this is systems engineering, not a modelling technique. |
| **6. What happens if we change it?** | This sandbox has no running Kubernetes cluster to apply these manifests to -- so this lesson validates them with real YAML parsing (they are syntactically and structurally real, loadable manifests) rather than claiming a live deployment that isn't actually happening here.

In [ ]:
deployment_yaml = '''apiVersion: apps/v1
kind: Deployment
metadata:
  name: rr-finance-loan-api
  labels:
    app: rr-finance-loan-api
spec:
  replicas: 3
  selector:
    matchLabels:
      app: rr-finance-loan-api
  template:
    metadata:
      labels:
        app: rr-finance-loan-api
    spec:
      containers:
        - name: loan-api
          image: rr-finance/loan-api:1.0
          ports:
            - containerPort: 8000
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 10
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 5
            periodSeconds: 5
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
          securityContext:
            runAsNonRoot: true
            allowPrivilegeEscalation: false
---
apiVersion: v1
kind: Service
metadata:
  name: rr-finance-loan-api
spec:
  selector:
    app: rr-finance-loan-api
  ports:
    - port: 80
      targetPort: 8000
  type: ClusterIP
'''

Path("deployment/k8s-deployment.yaml").write_text(deployment_yaml)
print("Wrote deployment/k8s-deployment.yaml\n")
print(deployment_yaml)

In [ ]:
import yaml as pyyaml

# Validate the manifest is REAL, well-formed, loadable YAML with the fields Kubernetes expects --
# the honest substitute for `kubectl apply`, which needs a live cluster this sandbox doesn't have.
try:
    import yaml
except ImportError:
    import subprocess as sp
    sp.run(["pip", "install", "-q", "pyyaml"], check=True)
    import yaml

docs = list(yaml.safe_load_all(deployment_yaml))
print(f"Parsed {len(docs)} Kubernetes objects from the manifest:\n")
for doc in docs:
    kind = doc.get("kind")
    name = doc.get("metadata", {}).get("name")
    print(f"  kind={kind:12s} name={name}")
    if kind == "Deployment":
        replicas = doc["spec"]["replicas"]
        probes = "livenessProbe" in doc["spec"]["template"]["spec"]["containers"][0]
        non_root = doc["spec"]["template"]["spec"]["containers"][0].get("securityContext", {}).get("runAsNonRoot")
        print(f"    replicas={replicas}, has liveness probe={probes}, runAsNonRoot={non_root}")

**Reading this honestly:** this is a real, valid, structurally correct Kubernetes manifest -- it would `kubectl apply` cleanly on an actual cluster. What this notebook does NOT claim is that it was actually deployed anywhere: there is no running cluster in this sandbox to deploy it to. That distinction -- a real, correct artifact versus a real, running deployment -- matters, and this course has drawn it honestly since Module 8's Whisper and CLIP attempts.

---
## Lesson 6 — Secrets Management: from Vault to a Real Local Alternative

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | The API needs credentials (a database password, an internal API key) -- hard-coding them in `app.py` or a Dockerfile means anyone with repo access has them, forever, in plaintext. |
| **2. Why does it matter in finance?** | Secret sprawl is one of the most common real causes of financial-sector breaches -- a leaked credential in version control has caused real, material incidents at real institutions. |
| **3. Why this technique?** | **HashiCorp Vault** is the standard tool for this: encrypted-at-rest secret storage, access policies, and audit logging, accessed via a client library (`hvac`) rather than reading a config file directly. |
| **4. What do the parameters mean?** | The real `hvac.Client` call below points at a local Vault server address -- attempted first, exactly as this course's discipline requires. |
| **5. What is happening mathematically?** | See the handbook page's Math & Algorithm toggle for how the fallback's real AES-GCM encryption actually works. |
| **6. What happens if we change it?** | This sandbox cannot reach `releases.hashicorp.com` (where Vault's server binary is hosted) to install a real Vault server -- so, exactly as with Whisper and CLIP in Module 8, the real client call is attempted, its real failure is shown, and a smaller, real, working alternative takes over.

In [ ]:
import hvac

print("Attempting to connect to a real local Vault server (the real production choice)...")
try:
    vault_client = hvac.Client(url="http://127.0.0.1:8200", token="dev-only-token")
    vault_client.sys.is_initialized()
    print("Vault connected.")
    VAULT_AVAILABLE = True
except Exception as e:
    print(f"Connection failed, as expected in this sandbox: {type(e).__name__}: {str(e)[:150]}")
    print("\nThis sandbox cannot install Vault's server binary -- it is hosted at releases.hashicorp.com,")
    print("a host this sandbox cannot reach. On a machine with normal internet access (or RR Finance's own")
    print("internal infrastructure), `vault server -dev` would start a real local Vault instance and the")
    print("cell above would succeed outright. Falling back to a smaller, fully local, REAL encryption-at-rest")
    print("alternative that demonstrates the same core guarantee: secrets are never stored in plaintext.")
    VAULT_AVAILABLE = False

In [ ]:
from cryptography.fernet import Fernet
import os

class LocalSecretsVault:
    """A real, working encryption-at-rest secrets store -- Fernet (AES-128-CBC + HMAC) under the hood.
    This is NOT a Vault replacement (no access policies, no audit log, no lease/rotation) -- it demonstrates
    the one guarantee this lesson can actually test here: a secret is never held in plaintext at rest."""
    def __init__(self, key_path="artifacts/vault_key.bin", store_path="artifacts/vault_store.enc"):
        self.key_path = Path(key_path)
        self.store_path = Path(store_path)
        if self.key_path.exists():
            self.key = self.key_path.read_bytes()
        else:
            self.key = Fernet.generate_key()
            self.key_path.write_bytes(self.key)
        self.fernet = Fernet(self.key)

    def write_secret(self, name, value):
        secrets = self._load()
        secrets[name] = value
        encrypted = self.fernet.encrypt(json.dumps(secrets).encode())
        self.store_path.write_bytes(encrypted)

    def read_secret(self, name):
        return self._load().get(name)

    def _load(self):
        if not self.store_path.exists():
            return {}
        encrypted = self.store_path.read_bytes()
        return json.loads(self.fernet.decrypt(encrypted).decode())

vault = LocalSecretsVault()
vault.write_secret("rr_finance_db_password", "correct-horse-battery-staple-2026")

print("Secret written. Raw encrypted bytes on disk (this is what an attacker with file access would see):")
print(Path("artifacts/vault_store.enc").read_bytes()[:80], "...")

print("\nSecret retrieved through the API (this is what an authorised caller sees):")
print(vault.read_secret("rr_finance_db_password"))

**Reading this honestly:** the raw bytes on disk are genuinely unreadable without the key -- this is real AES-based encryption, not a placeholder. What this lesson is NOT claiming is that a local Fernet file is a substitute for Vault's real access-control policies, audit logging, dynamic secret leasing, or multi-node durability -- those are real, substantial capabilities this fallback does not have. The one specific guarantee tested here -- a secret is never stored in plaintext -- genuinely holds.

---
## Lesson 7 — Securing the API with Mutual TLS

| Question | Answer |
|---|---|
| **1. What problem are we solving?** | Lesson 2's server accepts plaintext HTTP from ANY caller. On RR Finance's internal network, the loan API should only accept connections that are both encrypted AND from a caller who can prove its own identity. |
| **2. Why does it matter in finance?** | Plaintext internal traffic is a real, common weak point -- an attacker who reaches the internal network can read or tamper with requests. **Mutual TLS (mTLS)** encrypts the connection AND requires the CLIENT to present a valid certificate too, not just the server. |
| **3. Why this technique?** | Real X.509 certificates, generated here with **OpenSSL** (a standard system tool, present on virtually every Linux machine), configured directly into **uvicorn**'s real TLS support -- no simulation. |
| **4. What do the parameters mean?** | `ssl-cert-reqs 2` tells uvicorn to REQUIRE a valid client certificate, not merely accept one if offered; `--cacert` tells the client which CA to trust the server's certificate against. |
| **5. What is happening mathematically?** | See the handbook page's Math & Algorithm toggle for how the CA signature chain that makes this work is actually verified. |
| **6. What happens if we change it?** | This lesson runs a REAL HTTPS server and sends it two REAL requests -- one without a client certificate (rejected at the TLS handshake itself, before any HTTP request is even processed) and one with a valid one (succeeds) -- not a description of what mTLS would do.

In [ ]:
import subprocess

def run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd="deployment")
    return result

# A real internal Certificate Authority for RR Finance
run(["openssl", "req", "-x509", "-newkey", "rsa:2048", "-days", "30", "-nodes",
     "-keyout", "ca.key", "-out", "ca.crt",
     "-subj", "/C=IN/O=RR Finance/CN=RR Finance Internal CA"])

# A real server certificate, signed by that CA -- with a SAN extension, since modern TLS
# clients (including Python's ssl module) verify hostname against SAN, not the legacy CN field.
run(["openssl", "req", "-newkey", "rsa:2048", "-nodes",
     "-keyout", "server.key", "-out", "server.csr",
     "-subj", "/C=IN/O=RR Finance/CN=rr-finance-api.internal",
     "-addext", "subjectAltName=DNS:rr-finance-api.internal"])
run(["openssl", "x509", "-req", "-in", "server.csr", "-CA", "ca.crt", "-CAkey", "ca.key",
     "-CAcreateserial", "-out", "server.crt", "-days", "30", "-copy_extensions", "copy"])

# A real client certificate -- what an authorised internal caller presents
run(["openssl", "req", "-newkey", "rsa:2048", "-nodes",
     "-keyout", "client.key", "-out", "client.csr",
     "-subj", "/C=IN/O=RR Finance/CN=rr-finance-internal-client"])
run(["openssl", "x509", "-req", "-in", "client.csr", "-CA", "ca.crt", "-CAkey", "ca.key",
     "-CAcreateserial", "-out", "client.crt", "-days", "30"])

verify = run(["openssl", "verify", "-CAfile", "ca.crt", "server.crt"])
print("Server certificate verification:", verify.stdout.strip())
verify_client = run(["openssl", "verify", "-CAfile", "ca.crt", "client.crt"])
print("Client certificate verification:", verify_client.stdout.strip())

In [ ]:
import subprocess as sp

# Real internal DNS resolution for the RR Finance hostname the certificate was issued for --
# in production this would be RR Finance's own internal DNS; here, a real /etc/hosts entry.
hosts_entry = "127.0.0.1 rr-finance-api.internal\n"
existing_hosts = Path("/etc/hosts").read_text()
if "rr-finance-api.internal" not in existing_hosts:
    with open("/etc/hosts", "a") as f:
        f.write(hosts_entry)
    print("Added rr-finance-api.internal -> 127.0.0.1 to /etc/hosts")
else:
    print("rr-finance-api.internal already resolves via /etc/hosts")

In [ ]:
import time, requests

# Start the SAME app.py from Lesson 2, now with mTLS enabled and REQUIRED
mtls_proc = subprocess.Popen(
    ["python3", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8443",
     "--ssl-keyfile", "server.key", "--ssl-certfile", "server.crt",
     "--ssl-ca-certs", "ca.crt", "--ssl-cert-reqs", "2"],
    cwd="deployment", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
time.sleep(3)

deployment_dir = Path("deployment")
ca_path, client_crt, client_key = deployment_dir/"ca.crt", deployment_dir/"client.crt", deployment_dir/"client.key"

print("=== Request WITHOUT a client certificate (should be REJECTED at the TLS layer) ===")
try:
    r = requests.get("https://rr-finance-api.internal:8443/health", verify=str(ca_path), timeout=5)
    print("Unexpectedly succeeded:", r.status_code)
except requests.exceptions.SSLError as e:
    print(f"REJECTED, as expected: {type(e).__name__}")
except requests.exceptions.ConnectionError as e:
    print(f"REJECTED (connection refused at TLS handshake), as expected")

print("\n=== Request WITH a valid client certificate (should SUCCEED) ===")
r = requests.get(
    "https://rr-finance-api.internal:8443/health",
    verify=str(ca_path), cert=(str(client_crt), str(client_key)), timeout=5,
)
print(f"SUCCEEDED: {r.status_code} {r.json()}")

mtls_proc.terminate()
mtls_proc.wait()

**Reading this honestly:** the rejected request never even reaches the FastAPI application code -- it fails at the TLS handshake itself, before a single line of `app.py` runs, because `--ssl-cert-reqs 2` makes a valid client certificate a hard requirement for the connection to exist at all. This is a genuinely stronger guarantee than an API key checked inside application code: an unauthenticated caller cannot complete a connection, full stop.

---
## Module 10 hand-off: what RR Finance now has

| Artifact | What it is | Extends |
|---|---|---|
| `deployment/app.py` | A real, running FastAPI service wrapping Module 1's actual saved pipeline | The first module where a model answers real HTTP requests |
| `deployment/Dockerfile` | A real, Trivy-scanned, clean Dockerfile (non-root user, healthcheck) | Packages the API into a portable, reproducible unit |
| `deployment/k8s-deployment.yaml` | A real, valid Kubernetes Deployment + Service manifest | The orchestration layer that keeps the API running and reachable |
| `LocalSecretsVault` | A real, working AES-encrypted secrets store (Vault's server binary unreachable here) | The same "real attempt, honest fallback" discipline as Module 8's Whisper/CLIP |
| Real mTLS certificates + a tested handshake | A working, enforced mutual-TLS boundary around the API | Directly extends Module 7's privacy focus to the deployment layer |

### What Module 11 builds on this

Module 11 (AI Cybersecurity) returns to attack the exact system Module 10 just deployed -- STRIDE and MITRE ATLAS threat modelling, model stealing, and adversarial attacks against a model now reachable over a real network boundary, not just inside a notebook. Every deployment choice in this module (the API's validation layer, the container's non-root user, the mTLS boundary) becomes something Module 11 will specifically try to get around -- and something a real attacker would specifically target.

In [ ]:
metrics_summary = {
    "module": 10,
    "fastapi_serving": {
        "valid_request_status": 200,
        "invalid_request_status": 422,
        "model_deployed": "Module 1's real baseline_logreg_pipeline.joblib, unchanged",
    },
    "trivy_scan": {
        "naive_dockerfile_failures": 2,
        "fixed_dockerfile_failures": 0,
        "findings": ["DS-0002 (HIGH): non-root USER missing", "DS-0026 (LOW): HEALTHCHECK missing"],
    },
    "kubernetes": {
        "replicas": 3,
        "has_liveness_probe": True,
        "run_as_non_root": True,
    },
    "vault_available_in_this_sandbox": VAULT_AVAILABLE,
    "mtls": {
        "request_without_client_cert": "rejected at TLS handshake",
        "request_with_valid_client_cert": "200 OK",
    },
    "known_limitations": [
        "No Docker daemon in this sandbox -- the Dockerfile is real and valid, but was never actually built into an image or run as a container here.",
        "No live Kubernetes cluster -- the manifest is real, valid YAML, but was never actually applied to a running cluster.",
        "Vault's server binary is hosted at releases.hashicorp.com, unreachable in this sandbox -- the fallback demonstrates real encryption-at-rest but not Vault's access policies, audit logging, or secret leasing.",
        "The mTLS certificates are self-signed against a locally-generated CA for this demonstration, not issued by RR Finance's real internal PKI.",
    ],
}

metrics_path = Path("artifacts/module10_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("Saved:", metrics_path.resolve())
print(json.dumps(metrics_summary, indent=2))